In [ ]:
#Luis Jose Gonzalez Montilla
!pip install -q transformers datasets accelerate

import time
import numpy as np
from transformers import pipeline
from datasets import load_dataset

MAX_EXAMPLES = 200
TEXT_COL = "text"

def label_to_binary(label_str):
    ls = label_str.upper()
    if "POS" in ls: return 1
    if "NEG" in ls: return 0
    if "LABEL_1" in ls: return 1
    if "LABEL_0" in ls: return 0
    return 0

def extraer_label(salida):
    if isinstance(salida, list):
        item = salida[0]
        if isinstance(item, dict):
            return item["label"]
        if isinstance(item, list):
            return item[0]["label"]
    return "LABEL_0"

print("Cargando datasets en modo streaming...")

ds_medical = load_dataset(
    "FreedomIntelligence/medical-o1-reasoning-SFT",
    "en",
    split="train",
    streaming=True
)

ds_pdfs = load_dataset(
    "HuggingFaceFW/finepdfs",
    split="train",
    streaming=True
)

# Convertir streaming → lista limitada
def tomar_n_ejemplos(dataset, n):
    datos = []
    for i, item in enumerate(dataset):
        if i >= n:
            break
        datos.append(item)
    return datos

print("Extrayendo ejemplos...")
medical_samples = tomar_n_ejemplos(ds_medical, MAX_EXAMPLES)
pdf_samples = tomar_n_ejemplos(ds_pdfs, MAX_EXAMPLES)

# Unificar datasets
todos = []

# medical-o1 tiene campos: question, answer
for item in medical_samples:
    texto = item.get("question") or item.get("answer") or ""
    texto = texto.strip()
    if texto:
        todos.append({
            "text": texto[:800],
            "label": np.random.randint(0,2)
        })

# finepdfs tiene campos: text
for item in pdf_samples:
    texto = item.get("text") or ""
    texto = texto.strip()
    if texto:
        todos.append({
            "text": texto[:800],
            "label": np.random.randint(0,2)
        })

print(f"Total cargado: {len(todos)} ejemplos.\n")

# Evaluación

def evaluar_modelo(model_name, dataset):

    print(f"\n=== Evaluando: {model_name} ===")

    try:
        clf = pipeline(
            task="text-classification",
            model=model_name,
            tokenizer=model_name,
            device_map="auto",
            top_k=1
        )
    except Exception as e:
        print("Error cargando modelo:", e)
        return None

    y_true = []
    y_pred = []

    inicio = time.time()

    for item in dataset:
        txt = item["text"]
        y_true.append(item["label"])

        try:
            salida = clf(txt, truncation=True, max_length=256)
            label = extraer_label(salida)
        except:
            label = "LABEL_0"

        y_pred.append(label_to_binary(label))

    fin = time.time()

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    accuracy = (y_true == y_pred).mean()
    tiempo_total = fin - inicio
    tiempo_prom = tiempo_total / len(dataset)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Tiempo total: {tiempo_total:.2f}s")
    print(f"Tiempo por ejemplo: {tiempo_prom:.4f}s")

    return {
        "accuracy": accuracy,
        "tiempo_total": tiempo_total,
        "tiempo_prom": tiempo_prom
    }

# modelos

modelos = {
    "distilbert_small": "distilbert-base-uncased-finetuned-sst-2-english",
    "bert_base":        "j-hartmann/emotion-english-distilroberta-base",
    "roberta_base":     "Bhumika/roberta-base-finetuned-sst2",
    "roberta_large":    "philschmid/roberta-large-sst2",
}

resultados = {}

for alias, model in modelos.items():
    resultados[alias] = evaluar_modelo(model, todos)

# Benchmark final

print("\n=============== BENCHMARK FINAL ===============")
print(f"{'Modelo':20} | {'ACC':6} | {'Tiempo (s)':10} | {'Seg/Ejemplo'}")
print("-"*70)

for alias, info in resultados.items():
    if info:
        print(f"{alias:20} | {info['accuracy']:.4f} | {info['tiempo_total']:.2f} | {info['tiempo_prom']:.4f}")
    else:
      print(f"{alias:20} | ERROR")


Cargando datasets en modo streaming...


Resolving data files:   0%|          | 0/579 [00:00<?, ?it/s]

Extrayendo ejemplos...
Total cargado: 200 ejemplos.


=== Evaluando: distilbert-base-uncased-finetuned-sst-2-english ===


Device set to use cpu


Accuracy: 0.5550
Tiempo total: 88.98s
Tiempo por ejemplo: 0.4449s

=== Evaluando: j-hartmann/emotion-english-distilroberta-base ===


...

Accuracy: 0.5200
Tiempo total: 93.50s
Tiempo por ejemplo: 0.4675s

=== Evaluando: Bhumika/roberta-base-finetuned-sst2 ===


Device set to use cpu


Accuracy: 0.4950
Tiempo total: 186.33s
Tiempo por ejemplo: 0.9317s

=== Evaluando: philschmid/roberta-large-sst2 ===


Device set to use cpu


Accuracy: 0.5250
Tiempo total: 670.40s
Tiempo por ejemplo: 3.3520s

=============== BENCHMARK FINAL ===============
Modelo               | ACC    | Tiempo (s) | Seg/Ejemplo
----------------------------------------------------------------------
distilbert_small     | 0.5550 | 88.98 | 0.4449
bert_base            | 0.5200 | 93.50 | 0.4675
roberta_base         | 0.4950 | 186.33 | 0.9317
roberta_large        | 0.5250 | 670.40 | 3.3520
